# Notebook 02 — Data Cleaning & Preprocessing
**Decodelabs Internship | Week 2 | Task 2**

---
Here, I  systematically cleaned the raw dataset. This involves replacing `?` with NaN,
dropping columns with excessive missingness, removing clinically invalid rows, and
handling duplicate patient encounters.


In [5]:
import sys, os
# Add project root to Python path so we can import configs and src
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from configs.config import (
    RAW_FILE, IDS_MAP_FILE, INTERIM_FILE, PROCESSED_FILE,
    TRAIN_FILE, VAL_FILE, TEST_FILE,
    FIGURES_DIR, TABLES_DIR, PAPER_FIG_DIR, PAPER_TAB_DIR,
    RANDOM_SEED, TARGET_COL, PATIENT_ID_COL, MEDICATION_COLS,
    AGE_ORDER, icd9_to_category, COLORS, ensure_dirs
)
from src.plot_utils import set_plot_style, save_figure, save_table
ensure_dirs()
set_plot_style()
print("Project config loaded. Random seed:", RANDOM_SEED)

Project config loaded. Random seed: 42


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(RAW_FILE, low_memory=False)
original_shape = df.shape
print(f"Loaded raw data: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded raw data: 101,766 rows × 50 columns


## Replacing the '?' with NaN everywhere

In [7]:
before_counts = {col: (df[col] == "?").sum() 
                 for col in df.select_dtypes(include="object").columns
                 if (df[col] == "?").sum() > 0}

# Replace '?' with NaN across all columns
df.replace("?", np.nan, inplace=True)

print("Replaced '?' with NaN in all columns.")
print()
print("=== Columns affected ===")
for col, cnt in sorted(before_counts.items(), key=lambda x: -x[1]):
    print(f"  {col:35s}: {cnt:,} values replaced")

total_q = sum(before_counts.values())
print(f"\nTotal replacements: {total_q:,}")

Replaced '?' with NaN in all columns.

=== Columns affected ===
  weight                             : 98,569 values replaced
  medical_specialty                  : 49,949 values replaced
  payer_code                         : 40,256 values replaced
  race                               : 2,273 values replaced
  diag_3                             : 1,423 values replaced
  diag_2                             : 358 values replaced
  diag_1                             : 21 values replaced

Total replacements: 192,849


## Assessing missing values after replacement

In [8]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"count": missing, "percent": missing_pct})
missing_df = missing_df[missing_df["count"] > 0].sort_values("percent", ascending=False)

print("=== Missing Values Summary ===")
print(missing_df.to_string())
print(f"\nTotal missing values: {missing.sum():,}")

=== Missing Values Summary ===
                   count  percent
weight             98569    96.86
max_glu_serum      96420    94.75
A1Cresult          84748    83.28
medical_specialty  49949    49.08
payer_code         40256    39.56
race                2273     2.23
diag_3              1423     1.40
diag_2               358     0.35
diag_1                21     0.02

Total missing values: 374,017


## Dropping high-missingness columns

I dropped columns where the missingness rate is too high to be useful or where
the column is not relevant to the readmission prediction task.

**Decisions:**
- `weight` (~97% missing): Drop. Almost no usable data.
- `payer_code` (~40% missing): Drop. Not a clinical predictor; administrative.
- `medical_specialty` (~49% missing): Drop. Too many missing to impute reliably.
- `encounter_id`: Drop. This is just a row ID, not a clinical feature.

**Retain:**
- `patient_nbr`: Kept temporarily for patient-aware splitting; but will it remove before modelling.
- `A1Cresult`: The `NaN` is informative missingness, since ordering a lab test reflects a clinician's judgment about the patient. The `NAN` simply means the A1c test was never ordered, probably due to readmission risk.
- `max_glu_serum`: Same logic, "test not ordered" is informative clinical behavior, whether a clinician checked long-term glucose control is a plausible readmission signal.

In [9]:
cols_to_drop = ["weight", "payer_code", "medical_specialty", "encounter_id"]

for col in cols_to_drop:
    pct = df[col].isnull().mean() * 100 if col != "encounter_id" else 0
    print(f"  Dropping: {col:25s}  (missing: {pct:.1f}%)")

df.drop(columns=cols_to_drop, inplace=True)

print(f"\nShape after dropping columns: {df.shape[0]:,} rows × {df.shape[1]} columns")

  Dropping: weight                     (missing: 96.9%)
  Dropping: payer_code                 (missing: 39.6%)
  Dropping: medical_specialty          (missing: 49.1%)
  Dropping: encounter_id               (missing: 0.0%)

Shape after dropping columns: 101,766 rows × 46 columns


## Removing clinically invalid rows

Some `discharge_disposition_id` values indicate the patient died or was sent to hospice. These patients cannot be readmitted, so including them would distort the model. I removed them following standard clinical exclusion criteria.

In [10]:
# Expired (11), Hospice/medical (13), Hospice/home (14), Expired in medical facility (19), Expired (20), Expired (21)

EXCLUDE_DISCHARGE_IDS = [11, 13, 14, 19, 20, 21]  

n_before = len(df)
df = df[~df["discharge_disposition_id"].isin(EXCLUDE_DISCHARGE_IDS)]
n_removed = n_before - len(df)

print(f"Removed {n_removed:,} rows with hospice/expired discharge disposition.")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Removed 2,423 rows with hospice/expired discharge disposition.
Shape: 99,343 rows × 46 columns


## Handling gender column

Gender has values: `Male`, `Female`, `Unknown/Invalid`. `Unknown/Invalid` is a tiny fraction: I drop those rows.

In [11]:
print("=== Gender value counts ===")
print(df["gender"].value_counts())

n_before = len(df)
df = df[df["gender"].isin(["Male", "Female"])]
print(f"\nRemoved {n_before - len(df)} rows with Unknown/Invalid gender.")
print(f"Remaining: {len(df):,}")

=== Gender value counts ===
gender
Female             53454
Male               45886
Unknown/Invalid        3
Name: count, dtype: int64

Removed 3 rows with Unknown/Invalid gender.
Remaining: 99,340


## Handling remaining missing values

 After the column drops and row removals, I checked what's left. For the diagnosis columns (diag_1, diag_2, diag_3), a small number of rows have missing values. I fill them with 'Unknown' since they will be categorised in the feature engineering step.

In [12]:
# Fill missing diagnosis codes with 'Unknown'
for col in ["diag_1", "diag_2", "diag_3"]:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        df[col].fillna("Unknown", inplace=True)
        print(f"  {col}: filled {n_missing} missing values with 'Unknown'")

# Check remaining missing values
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) == 0:
    print("\nNo remaining missing values. Dataset is complete.")
else:
    print("\nRemaining missing values:")
    print(remaining_missing.to_string())

  diag_1: filled 20 missing values with 'Unknown'
  diag_2: filled 356 missing values with 'Unknown'
  diag_3: filled 1419 missing values with 'Unknown'

Remaining missing values:
race              2232
max_glu_serum    94188
A1Cresult        82506


C:\Users\Peter\AppData\Local\Temp\ipykernel_16472\1984614637.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("Unknown", inplace=True)


## Creating binary target variable

In [13]:
# I binarised the readmitted column:
#   '<30' (early readmission) → 1
#   'NO' and '>30'            → 0

df["readmitted_binary"] = (df["readmitted"] == "<30").astype(int)

print("=== Original readmitted distribution ===")
print(df["readmitted"].value_counts())
print()
print("=== Binary target distribution ===")
counts = df["readmitted_binary"].value_counts().sort_index()
pcts   = df["readmitted_binary"].value_counts(normalize=True).sort_index() * 100
for v in [0, 1]:
    label = "No early readmission" if v == 0 else "Early readmission"
    print(f"  {v} ({label:25s}): {counts[v]:7,} ({pcts[v]:5.1f}%)")

# Drop the original readmitted column now that we have the binary version
df.drop(columns=["readmitted"], inplace=True)

=== Original readmitted distribution ===
readmitted
NO     52524
>30    35502
<30    11314
Name: count, dtype: int64

=== Binary target distribution ===
  0 (No early readmission     ):  88,026 ( 88.6%)
  1 (Early readmission        ):  11,314 ( 11.4%)


## Handling duplicate patient encounters

Multiple encounters per patient can distort training if not handled. The strategy is to keep only the FIRST encounter for each patient.

In [14]:
n_before    = len(df)
n_patients  = df[PATIENT_ID_COL].nunique()

# Sort by patient, then keep the first row per patient
df = df.sort_values([PATIENT_ID_COL]).drop_duplicates(subset=PATIENT_ID_COL, keep="first")
n_after     = len(df)

print(f"Rows before deduplication: {n_before:,}")
print(f"Rows after  deduplication: {n_after:,}")
print(f"Rows removed             : {n_before - n_after:,}")
print(f"Unique patients remaining: {df[PATIENT_ID_COL].nunique():,}")

Rows before deduplication: 99,340
Rows after  deduplication: 69,987
Rows removed             : 29,353
Unique patients remaining: 69,987


I keep only the first encounter per patient to avoid data leakage and to match the realistic clinical use case.

## Final cleaning summary

In [15]:
print("=" * 60)
print("  Data Cleaning Summary")
print("=" * 60)
print(f"  Original shape : {original_shape[0]:,} rows × {original_shape[1]} cols")
print(f"  Final shape    : {df.shape[0]:,} rows × {df.shape[1]} cols")
print()
print(f"  Rows removed:")
print(f"    Hospice/expired discharge  : ~{n_removed:,}")
print(f"    Unknown/Invalid gender     : small")
print(f"    Duplicate patients         : {n_before - n_after:,}")
print()
print(f"  Columns removed: weight, max_glu_serum, A1Cresult, payer_code, medical_specialty, encounter_id")
print(f"  Missing '?'    : replaced with NaN; diag missing filled with 'Unknown'")
print(f"  Target column  : readmitted → readmitted_binary (0 / 1)")
print()
print(f"  Missing values remaining: {df.isnull().sum().sum()}")
print(f"  Duplicate rows           : {df.duplicated().sum()}")

  Data Cleaning Summary
  Original shape : 101,766 rows × 50 cols
  Final shape    : 69,987 rows × 46 cols

  Rows removed:
    Hospice/expired discharge  : ~2,423
    Unknown/Invalid gender     : small
    Duplicate patients         : 29,353

  Columns removed: weight, max_glu_serum, A1Cresult, payer_code, medical_specialty, encounter_id
  Missing '?'    : replaced with NaN; diag missing filled with 'Unknown'
  Target column  : readmitted → readmitted_binary (0 / 1)

  Missing values remaining: 126027
  Duplicate rows           : 0


## Saving interim dataset

In [16]:
os.makedirs(os.path.dirname(INTERIM_FILE), exist_ok=True)
df.to_csv(INTERIM_FILE, index=False)
print(f"Interim dataset saved: {INTERIM_FILE}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Interim dataset saved: c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\data\interim\diabetic_interim.csv
Shape: 69,987 rows × 46 columns
